# Week 6: Week types (K-means + PCA)

Goal: group your weeks into a few **types** (for example *Big shopping week* or *Quiet week*) with **K-means**, then draw them in 2D with **PCA**. You name the types; the app then tells you which type this week looks like.

This is **unsupervised** learning: there are no labels to score against. You judge clusters by how separated they are (silhouette) and by whether they mean something.

Spec: `docs/superpowers/specs/2026-10-23-week-types-design.md`. Data: `data/fake_transactions_3y.csv` (from Week 5).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

CATEGORIES = ['Food & Drinks', 'Coffee & Snacks', 'Groceries', 'Transportation', 'Bills & Utilities',
              'Shopping', 'Healthcare', 'Entertainment']
FEATURES = [f'share_{c}' for c in CATEGORIES] + ['total_ratio']   # the app builds exactly these, in this order

df = pd.read_csv('../data/fake_transactions_3y.csv')
exp = df[df['Type'] == 'expense'].copy()
exp['day'] = pd.to_datetime(exp['Date']).dt.tz_localize(None).dt.normalize()
exp['week'] = exp['day'].dt.to_period('W-SUN')   # weeks start on Monday

# Whole weeks only: the file starts and ends mid-week
FIRST_WEEK, LAST_WEEK = exp['week'].min() + 1, exp['week'].max() - 1
print(FIRST_WEEK, '→', LAST_WEEK)

---
## A. Weekly table
One row per week (every week from `FIRST_WEEK` to `LAST_WEEK`, empty weeks = 0), one column per category in `CATEGORIES` order. Save it as `weekly`, and the week totals as `total`.

Hints: `pivot_table(index='week', columns='Category', values='Amount', aggfunc='sum', fill_value=0)`, then `reindex(pd.period_range(FIRST_WEEK, LAST_WEEK, freq='W-SUN'), fill_value=0)` and `reindex(columns=CATEGORIES, fill_value=0)`.

In [ ]:
# TODO


---
## B. Features
For each week, 9 numbers. Save them as a DataFrame `X` with columns `FEATURES`:
- `share_<category>`: that category's total ÷ the week's total (the 8 shares add up to 1). Hint: `weekly.div(total, axis=0).add_prefix('share_')`
- `total_ratio`: the week's total ÷ the **median of the weeks before it** (up to 8, at least 4). Hint: `total.shift().rolling(8, min_periods=4).median()`. `shift()` keeps the week itself out: no look-ahead.

Drop weeks with a total of 0 or without a ratio (`replace([np.inf, -np.inf], np.nan).dropna()`).

**Think:** why shares and a ratio instead of pesos? (Hint: the app will use these centres for real users who spend more or less than the fake data.)

In [ ]:
# TODO


This check recomputes one week straight from the raw expenses. If it passes, the features are right and there is no look-ahead.

In [ ]:
w = X.index[len(X) // 2]
rows = exp[exp['week'] == w]
assert abs(X.loc[w, 'share_Groceries'] - rows.loc[rows['Category'] == 'Groceries', 'Amount'].sum() / rows['Amount'].sum()) < 1e-9
before = [exp.loc[exp['week'] == w - k, 'Amount'].sum() for k in range(1, 9)]
assert abs(X.loc[w, 'total_ratio'] - rows['Amount'].sum() / np.median(before)) < 1e-9
assert list(X.columns[:9]) == FEATURES
print(len(X), 'weeks ✓')

---
## C. Scale
Fit a `StandardScaler` on `X[FEATURES]` (save it as `scaler`) and make `Xs = scaler.transform(X[FEATURES])`.

**Think:** K-means measures plain distance. What would happen without scaling, when `total_ratio` swings between 0.3 and 3 but a small share moves between 0.00 and 0.05?

In [ ]:
# TODO


---
## D. Choose k
For k = 2…8, fit `KMeans(k, n_init=10, random_state=0)` on `Xs` and record `inertia_` and `silhouette_score(Xs, labels_)`. Plot both against k.

- **Elbow:** where adding clusters stops helping much
- **Silhouette:** −1…1, higher = better separated

**Think:** if the silhouette is only ~0.2 for every k, what does that tell you about spending? Is that a reason not to use clusters?

In [ ]:
# TODO


Set `K` to your choice and write one sentence why below it.

In [ ]:
# TODO
K = ...


---
## E. Read the clusters
Fit the final `km = KMeans(K, n_init=10, random_state=0)` on `Xs`, add `X['cluster'] = km.labels_`, and show:
1. how many weeks each cluster has
2. the mean of each feature per cluster (unscaled: `X.groupby('cluster')[FEATURES].mean()`)

In [ ]:
# TODO


And the 3 real weeks nearest each centre (`km.transform(Xs)` gives distances). Look at what happened in those weeks.

In [ ]:
# TODO


Now **name the types**: one entry per cluster, **in cluster order** (0, 1, 2…). Each `name` ends in "week" (Ollie says *"So far this looks like a {name}."*), `description` is one short sentence, and `pose` is `'happy'`, `'thinking'` or `'celebrating'` (how should Ollie feel about that kind of week?).

In [ ]:
# TODO
TYPES = [
    {'name': '... week', 'description': '...', 'pose': '...'},
]


---
## F. PCA: see it in 2D
9 features can't be drawn. PCA finds the 2 directions with the most variation.
- `pca = PCA(2).fit(Xs)`, `P = pca.transform(Xs)`, and the centres `pca.transform(km.cluster_centers_)`
- Scatter `P`, coloured by cluster and labelled with your type names; mark the centres
- Put `pca.explained_variance_ratio_` in the axis labels
- Show the **loadings**: `pd.DataFrame(pca.components_.T, index=FEATURES, columns=['PC1', 'PC2'])`

**Think:** read the loadings. What does a week far to the right on PC1 look like? And high on PC2? PCA is only for the picture: K-means used all 9 features.

In [ ]:
# TODO


**What PC1 and PC2 mean:** TODO

---
## G. Export (given)
Writes `models/week_types.json` for the app: the scaler, the centres, your names, and 3 sample weeks the app must classify the same way.

In [ ]:
assert len(TYPES) == K, 'one TYPES entry per cluster'
for t in TYPES:
    assert t['name'].endswith('week') and t['description'] and t['pose'] in ('happy', 'thinking', 'celebrating'), t

dist = km.transform(Xs)
samples = []
for c in range(min(3, K)):
    i = int(dist[:, c].argmin())
    assert km.labels_[i] == c
    samples.append({'features': X[FEATURES].iloc[i].tolist(), 'type': f't{c}'})

model = {'features': FEATURES, 'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist(),
         'centers': km.cluster_centers_.tolist(),
         'types': [{'id': f't{i}', **t} for i, t in enumerate(TYPES)], 'samples': samples}
Path('../models').mkdir(exist_ok=True)
Path('../models/week_types.json').write_text(json.dumps(model, indent=1))
print('saved models/week_types.json ✓  (copy it to budgettracker/mobile/src/lib/week-types.json)')